In [0]:
%pip install databricks-vectorsearch mlflow langchain -q
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow.deployments
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import *

# Connect everything
client = mlflow.deployments.get_deploy_client("databricks")

vsc = VectorSearchClient(disable_notice=True)

index = vsc.get_index(
    endpoint_name="healthcare_vector_search_endpoint",
    index_name="workspace.default.healthcare_vector_index"
)

df = spark.table("healthcare_facilities_cleaned")
df.createOrReplaceTempView("facilities")

print("✅ All connections ready")
print(f"   LLM client: connected")
print(f"   Vector index: connected")
print(f"   Facilities table: {df.count()} rows")

✅ All connections ready
   LLM client: connected
   Vector index: connected
   Facilities table: 10000 rows


In [0]:

# AGENT 1: QUERY AGENT — Parses user question into structured intent


import json

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

def query_agent(user_question):
    """Takes natural language question, returns structured query intent"""
    
    prompt = """You are a healthcare facility search assistant for India.
Parse the user's question into a structured JSON query.

Extract these fields:
- "search_text": the core medical need (for semantic search)
- "state": Indian state filter if mentioned (null if not)
- "city": city filter if mentioned (null if not)  
- "facility_type": hospital/clinic/dentist if mentioned (null if not)
- "required_capabilities": list from [emergency, surgery, icu, dialysis, neonatal]
- "specialties_needed": list of medical specialties mentioned
- "min_trust_score": set to 60 if user wants reliable/trusted facilities, else 0
- "urgency": high/medium/low based on question tone

Respond with ONLY valid JSON. No explanation, no markdown, no backticks.

User Question: """ + user_question

    response = client.predict(
        endpoint=LLM_ENDPOINT,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 500,
            "temperature": 0.1
        }
    )
    
    try:
        result_text = response['choices'][0]['message']['content']
        result_text = result_text.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(result_text)
        print("🤖 Query Agent output:")
        print(json.dumps(parsed, indent=2))
        return parsed
    
    except Exception as e:
        print(f"⚠️ Parse error: {e}")
        return {
            "search_text": user_question,
            "state": None, "city": None, "facility_type": None,
            "required_capabilities": [], "specialties_needed": [],
            "min_trust_score": 0, "urgency": "medium"
        }

# TEST
print("=" * 60)
query_agent("Find the nearest facility in rural Bihar that can perform an emergency appendectomy")

🤖 Query Agent output:
{
  "search_text": "emergency appendectomy",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [
    "general surgery"
  ],
  "min_trust_score": 60,
  "urgency": "high"
}


{'search_text': 'emergency appendectomy',
 'state': 'Bihar',
 'city': None,
 'facility_type': None,
 'required_capabilities': ['emergency', 'surgery'],
 'specialties_needed': ['general surgery'],
 'min_trust_score': 60,
 'urgency': 'high'}

In [0]:

# AGENT 2: SEARCH AGENT — Vector search + structured filters


def search_agent(query_intent, top_k=10):
    """Takes parsed intent, searches vector index + applies filters, returns candidates"""
    
    search_text = query_intent.get("search_text", "")
    state = query_intent.get("state")
    city = query_intent.get("city")
    facility_type = query_intent.get("facility_type")
    required_caps = query_intent.get("required_capabilities", [])
    min_trust = query_intent.get("min_trust_score", 0)
    
    # Step 1: Embed the search text
    query_embedding = client.predict(
        endpoint="databricks-bge-large-en",
        inputs={"input": [search_text]}
    ).data[0]["embedding"]
    
    # Step 2: Vector search
    vector_results = index.similarity_search(
        query_vector=query_embedding,
        num_results=50,
        columns=["id", "name_cleaned", "combined_text"]
    )
    
    # Get matching IDs from vector search
    matches = vector_results.get("result", {}).get("data_array", [])
    matched_ids = [int(row[0]) for row in matches]
    scores = {int(row[0]): float(row[3]) for row in matches}
    
    if not matched_ids:
        print("⚠️ No vector search results found")
        return []
    
    # Step 3: Apply structured filters on the main table
    sql_filters = ["1=1"]
    
    if state:
        sql_filters.append(f"state = '{state}'")
    if city:
        sql_filters.append(f"city = '{city}'")
    if facility_type:
        sql_filters.append(f"facilityType = '{facility_type}'")
    if min_trust > 0:
        sql_filters.append(f"trust_score >= {min_trust}")
    
    # Capability filters
    cap_map = {
        "emergency": "has_emergency = true",
        "surgery": "has_surgery = true",
        "icu": "has_icu = true",
        "dialysis": "has_dialysis = true",
        "neonatal": "has_neonatal = true"
    }
    for cap in required_caps:
        if cap in cap_map:
            sql_filters.append(cap_map[cap])
    
    where_clause = " AND ".join(sql_filters)
    
    # Query the facilities table with filters
    query = f"""
        SELECT 
            name_cleaned, city, state, facilityType,
            trust_score, trust_label, completeness_score,
            has_emergency, has_surgery, has_icu, has_dialysis, has_neonatal,
            description_cleaned, combined_text,
            flag_surgery_no_surgeon, flag_icu_no_equipment, 
            flag_emergency_no_basics, flag_specialty_no_specialist,
            latitude_valid, longitude_valid
        FROM facilities
        WHERE {where_clause}
        ORDER BY trust_score DESC
        LIMIT {top_k}
    """
    
    filtered_results = spark.sql(query).collect()
    
    # Combine results
    results = []
    for row in filtered_results:
        results.append({
            "facility_name": row["name_cleaned"],
            "city": row["city"],
            "state": row["state"],
            "facility_type": row["facilityType"],
            "trust_score": row["trust_score"],
            "trust_label": row["trust_label"],
            "completeness_score": row["completeness_score"],
            "has_emergency": row["has_emergency"],
            "has_surgery": row["has_surgery"],
            "has_icu": row["has_icu"],
            "has_dialysis": row["has_dialysis"],
            "has_neonatal": row["has_neonatal"],
            "description": row["description_cleaned"],
            "combined_text": row["combined_text"],
            "flags": {
                "surgery_no_surgeon": row["flag_surgery_no_surgeon"],
                "icu_no_equipment": row["flag_icu_no_equipment"],
                "emergency_no_basics": row["flag_emergency_no_basics"],
                "specialty_no_specialist": row["flag_specialty_no_specialist"]
            },
            "latitude": row["latitude_valid"],
            "longitude": row["longitude_valid"]
        })
    
    print(f"🔍 Search Agent found {len(results)} facilities")
    print(f"   Filters applied: {where_clause}")
    
    for i, r in enumerate(results[:5]):
        print(f"\n   {i+1}. {r['facility_name']} ({r['city']}, {r['state']})")
        print(f"      Trust: {r['trust_score']} ({r['trust_label']}) | Emergency: {r['has_emergency']} | Surgery: {r['has_surgery']}")
    
    return results

# TEST
intent = query_agent("Find the nearest facility in rural Bihar that can perform an emergency appendectomy")
print("\n" + "=" * 60)
candidates = search_agent(intent)

🤖 Query Agent output:
{
  "search_text": "emergency appendectomy",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [
    "general surgery"
  ],
  "min_trust_score": 60,
  "urgency": "high"
}

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 4 facilities
   Filters applied: 1=1 AND state = 'Bihar' AND trust_score >= 60 AND has_emergency = true AND has_surgery = true

   1. Joint & Sports Injury Clinic (Patna, Bihar)
      Trust: 75 (GREEN) | Emergency: True | Surgery: True

   2. Jalal Medical Center (Motihari, Bihar)
      Trust: 75 (GREEN) | Emergency: True | Surgery: True

   3. A.R DENTAL EXPERT - Kishanganj Branch (Kishanganj, Bihar)
      Trust: 75 (GREEN) | Emergency: True | Surgery: True

   4. Shree

In [0]:


# AGENT 3 (FINAL): CareGuide — Confident, verified recommendations


def validator_agent(user_question, query_intent, candidates):
    """CareMap's CareGuide - gives confident, verified recommendations"""
    
    if not candidates:
        return {
            "answer": "I wasn't able to find facilities matching your exact needs right now. If this is an emergency, please call 108 (India Emergency Helpline) immediately. You can also try broadening your search to nearby districts or states. — CareGuide, by CareMap",
            "validated_results": [],
            "reasoning_steps": []
        }
    
    facility_summaries = ""
    for i, c in enumerate(candidates[:5]):
        flags_triggered = [k for k, v in c["flags"].items() if v]
        flag_text = ", ".join(flags_triggered) if flags_triggered else "None"
        
        facility_summaries += f"""
FACILITY {i+1}: {c['facility_name']}
- Location: {c['city']}, {c['state']}
- Type: {c['facility_type']}
- Trust Score: {c['trust_score']}/100 ({c['trust_label']})
- Completeness: {c['completeness_score']}%
- Emergency: {c['has_emergency']} | Surgery: {c['has_surgery']} | ICU: {c['has_icu']}
- Dialysis: {c['has_dialysis']} | Neonatal: {c['has_neonatal']}
- Flags Triggered: {flag_text}
- Description: {c['description'][:400] if c['description'] else 'No description'}
- Coordinates: {c['latitude']}, {c['longitude']}
"""
    
    urgency = query_intent.get("urgency", "medium")
    
    prompt = f"""You are CareGuide, the intelligent assistant behind CareMap — a healthcare discovery platform that has already verified 10,000+ medical facilities across India. you have to greet visitors and provide helpful, accurate, personalized healthcare recommendations.

You have a list of facilities that match the user's search intent. You have already checked and verified each facility's equipment, staff, capabilities, and flagged any contradictions. You are confident in your recommendations. DO NOT ask the user to "call and verify" or "confirm before going" — that defeats your purpose. You ARE the verification layer.

Your personality: confident, warm, knowledgeable. You speak like a doctor friend who has personally checked these hospitals and is giving trusted advice.

The user asked: "{user_question}"
Urgency level: {urgency}

Facilities found and verified:

IMPORTANT: You have ALREADY checked and verified each facility's equipment, staff, capabilities, and flagged any contradictions. You are confident in your recommendations. DO NOT ask the user to "call and verify" or "confirm before going" — that defeats your purpose. You ARE the verification layer.

Your personality: confident, warm, knowledgeable. You speak like a doctor friend who has personally checked these hospitals and is giving trusted advice.

The user asked: "{user_question}"
Urgency level: {urgency}

Facilities found and verified:
{facility_summaries}

Rules:
1. ANSWER (for the user):
   - If urgent, acknowledge urgency and mention 108 helpline
   - Give a confident, clear recommendation — "Go to X in Y city"
   - Explain what the facility offers that matches their need
   - If a facility has concerns, say it directly: "I'd avoid X because..."
   - NEVER mention trust scores, flags, percentages, or technical terms
   - NEVER ask the user to verify anything — YOU already did that
   - Sign off: "— CareGuide, by CareMap"

2. FACILITY CARDS (for frontend):
   - verdict: VERIFIED & RECOMMENDED / USE WITH CAUTION / NOT SUITABLE
   - what_they_offer: specific services this facility has for the user's need
   - concerns: honest assessment if any, in simple language (e.g. "This facility lists surgery services but we couldn't confirm a surgeon on staff")
   - If no concerns, say "Verified — no issues found"

3. CHAIN OF THOUGHT (for transparency panel):
   - Full technical reasoning with trust scores, flags, data points
   - This is for technical audience, be detailed

JSON format:
{{
    "answer": "Confident, warm recommendation (3-5 sentences). — CareGuide, by CareMap",
    "emergency_note": "Call 108 immediately if life-threatening" or null,
    "validated_results": [
        {{
            "rank": 1,
            "facility_name": "name",
            "city": "city",
            "state": "state",
            "latitude": 0.0,
            "longitude": 0.0,
            "verdict": "VERIFIED & RECOMMENDED / USE WITH CAUTION / NOT SUITABLE",
            "what_they_offer": "What this facility specifically has for the user's need",
            "concerns": "Any honest concerns OR 'Verified — no issues found'"
        }}
    ],
    "reasoning_steps": [
        "Step 1: ...",
        "Step 2: ...",
        "Step 3: ..."
    ]
}}

ONLY valid JSON. No markdown, no backticks."""

    response = client.predict(
        endpoint=LLM_ENDPOINT,
        inputs={
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": 2000,
            "temperature": 0.3
        }
    )
    
    try:
        result_text = response['choices'][0]['message']['content']
        result_text = result_text.replace("```json", "").replace("```", "").strip()
        validated = json.loads(result_text)
        
        print("💬 CareGuide:")
        print(f"\n   {validated['answer']}")
        
        if validated.get("emergency_note"):
            print(f"\n   🚨 {validated['emergency_note']}")
        
        print(f"\n📋 Verified Facility Cards:")
        for r in validated.get("validated_results", []):
            emoji = "✅" if "RECOMMENDED" in r["verdict"] else "⚠️" if "CAUTION" in r["verdict"] else "❌"
            print(f"\n   {emoji} #{r['rank']} {r['facility_name']} — {r['city']}, {r['state']}")
            print(f"      Status: {r['verdict']}")
            print(f"      Offers: {r['what_they_offer']}")
            print(f"      Assessment: {r['concerns']}")
        
        print(f"\n🧠 Chain of Thought:")
        for step in validated.get("reasoning_steps", []):
            print(f"   → {step}")
        
        return validated
    except Exception as e:
        print(f"⚠️ Error: {e}")
        return {
            "answer": "I found some facilities but had trouble processing the details. Please try again. — CareGuide, by CareMap",
            "validated_results": [],
            "reasoning_steps": [f"Error: {str(e)}"]
        }



In [0]:
def healthcare_agent(user_question):
    """CareMap — Complete agentic pipeline"""
    
    print("🗺️  C A R E M A P")
    print("=" * 60)
    print(f"📝 {user_question}")
    print("=" * 60)
    
    print("\n🔵 STEP 1: Understanding your question...")
    print("-" * 40)
    intent = query_agent(user_question)
    
    print("\n🟢 STEP 2: Searching facilities across India...")
    print("-" * 40)
    candidates = search_agent(intent)
    
    print("\n🟡 STEP 3: CareGuide is reviewing results...")
    print("-" * 40)
    validated = validator_agent(user_question, intent, candidates)
    
    print("\n" + "=" * 60)
    print("🗺️  CareMap — Your bridge to better healthcare")
    print("=" * 60)
    
    return {
        "question": user_question,
        "intent": intent,
        "candidates_found": len(candidates),
        "candidates": candidates,
        "validation": validated
    }


# ============================================================================
# TEST
# ============================================================================
result = healthcare_agent("Find the nearest facility in rural Bihar that can perform an emergency appendectomy")

🗺️  C A R E M A P
📝 Find the nearest facility in rural Bihar that can perform an emergency appendectomy

🔵 STEP 1: Understanding your question...
----------------------------------------
🤖 Query Agent output:
{
  "search_text": "emergency appendectomy",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [
    "general surgery"
  ],
  "min_trust_score": 60,
  "urgency": "high"
}

🟢 STEP 2: Searching facilities across India...
----------------------------------------
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 4 facilities
   Filters applied: 1=1 AND state = 'Bihar' AND trust_score >= 60 AND has_emergency = true AND has_surgery = true

   1. Joint & Sports Injury Clinic (Patna, Bihar)
      Trust: 75 (GREEN

In [0]:
# ============================================================================
# TEST SUITE — Different query types for demo
# ============================================================================

test_queries = [
    "Which districts in Uttar Pradesh have no dialysis center?",
    "Show me trusted eye hospitals in Hyderabad with good ratings",
    "My mother needs neonatal care in Rajasthan urgently"
]

for q in test_queries:
    result = healthcare_agent(q)
    print("\n\n")

🗺️  C A R E M A P
📝 Which districts in Uttar Pradesh have no dialysis center?

🔵 STEP 1: Understanding your question...
----------------------------------------
🤖 Query Agent output:
{
  "search_text": "dialysis center",
  "state": "Uttar Pradesh",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "dialysis"
  ],
  "specialties_needed": [],
  "min_trust_score": 0,
  "urgency": "low"
}

🟢 STEP 2: Searching facilities across India...
----------------------------------------
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 10 facilities
   Filters applied: 1=1 AND state = 'Uttar Pradesh' AND has_dialysis = true

   1. Kanishk Homoeopathy Centre (Modinagar, Uttar Pradesh)
      Trust: 100 (GREEN) | Emergency: False | Surgery: False

   2. Memansha Medical Store (Lucknow, Uttar Pradesh)
 

In [0]:
def caremap_api(user_question):
    """Single function — complete CareMap response"""
    result = healthcare_agent(user_question)
    
    api_response = {
        # User's question
        "question": result["question"],
        
        # Parsed intent from Query Agent
        "intent": result["intent"],
        
        # How many facilities matched
        "candidates_found": result["candidates_found"],
        
        # CareGuide's warm message to user
        "careguide_message": result["validation"].get("answer", ""),
        
        # Emergency helpline note (null if not urgent)
        "emergency_note": result["validation"].get("emergency_note"),
        
        # Facility cards with verdict, services, concerns
        "facility_cards": result["validation"].get("validated_results", []),
        
        # Chain of thought steps for transparency panel
        "chain_of_thought": result["validation"].get("reasoning_steps", []),
        
        # Raw facility data with coordinates for map + technical details
        "facilities": []
    }
    
    # Add full facility details for map and technical cards
    for c in result["candidates"]:
        api_response["facilities"].append({
            "name": c["facility_name"],
            "city": c["city"],
            "state": c["state"],
            "facility_type": c["facility_type"],
            "lat": c["latitude"],
            "lng": c["longitude"],
            "trust_score": c["trust_score"],
            "trust_label": c["trust_label"],
            "completeness_score": c["completeness_score"],
            "description": c["description"],
            "has_emergency": c["has_emergency"],
            "has_surgery": c["has_surgery"],
            "has_icu": c["has_icu"],
            "has_dialysis": c["has_dialysis"],
            "has_neonatal": c["has_neonatal"],
            "flags": c["flags"]
        })
    
    return api_response

# Test and print full response
response = caremap_api("Find emergency surgery facility in Bihar")
print(json.dumps(response, indent=2, default=str))

🗺️  C A R E M A P
📝 Find emergency surgery facility in Bihar

🔵 STEP 1: Understanding your question...
----------------------------------------
🤖 Query Agent output:
{
  "search_text": "emergency surgery",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [],
  "min_trust_score": 60,
  "urgency": "high"
}

🟢 STEP 2: Searching facilities across India...
----------------------------------------
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 4 facilities
   Filters applied: 1=1 AND state = 'Bihar' AND trust_score >= 60 AND has_emergency = true AND has_surgery = true

   1. Joint & Sports Injury Clinic (Patna, Bihar)
      Trust: 75 (GREEN) | Emergency: True | Surgery: True

   2. Jalal Medical Center (Motihari

In [0]:
response = caremap_api("Find emergency surgery facility in Bihar")
print(json.dumps(response, indent=2, default=str))

🗺️  C A R E M A P
📝 Find emergency surgery facility in Bihar

🔵 STEP 1: Understanding your question...
----------------------------------------
🤖 Query Agent output:
{
  "search_text": "emergency surgery",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [],
  "min_trust_score": 60,
  "urgency": "high"
}

🟢 STEP 2: Searching facilities across India...
----------------------------------------
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 4 facilities
   Filters applied: 1=1 AND state = 'Bihar' AND trust_score >= 60 AND has_emergency = true AND has_surgery = true

   1. Joint & Sports Injury Clinic (Patna, Bihar)
      Trust: 75 (GREEN) | Emergency: True | Surgery: True

   2. Jalal Medical Center (Motihari

In [0]:
import json

demo_queries = [
    "Find the nearest facility in rural Bihar that can perform an emergency appendectomy",
    "Show me trusted eye hospitals in Hyderabad",
    "My mother needs neonatal care in Rajasthan urgently",
    "Find dialysis centers in Uttar Pradesh",
    "Which hospitals in Maharashtra have ICU facilities",
    "Find a dental clinic in Delhi with good trust score"
]

demo_responses = {}

for q in demo_queries:
    print(f"\n🔄 Processing: {q[:60]}...")
    try:
        result = caremap_api(q)
        demo_responses[q] = result
        print(f"   ✅ Done — {result['candidates_found']} facilities found")
    except Exception as e:
        print(f"   ❌ Error: {str(e)[:100]}")

with open("/Workspace/Users/umarofficial56@gmail.com/caremap_demo_responses.json", "w") as f:
    json.dump(demo_responses, f, indent=2, default=str)

print(f"\n✅ Saved {len(demo_responses)} responses — DOWNLOAD AND SEND TO TASEER")


🔄 Processing: Find the nearest facility in rural Bihar that can perform an...
🗺️  C A R E M A P
📝 Find the nearest facility in rural Bihar that can perform an emergency appendectomy

🔵 STEP 1: Understanding your question...
----------------------------------------
🤖 Query Agent output:
{
  "search_text": "emergency appendectomy",
  "state": "Bihar",
  "city": null,
  "facility_type": null,
  "required_capabilities": [
    "emergency",
    "surgery"
  ],
  "specialties_needed": [
    "general surgery"
  ],
  "min_trust_score": 60,
  "urgency": "high"
}

🟢 STEP 2: Searching facilities across India...
----------------------------------------
[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
🔍 Search Agent found 4 facilities
   Filters applied: 1=1 AND state = 'Bihar' AND trust_score >= 60 AND has_emergency = true AND has_surgery =

In [0]:
# ============================================================================
# FIXED MLFLOW MODEL - Works in Model Serving (No PySpark)
# ============================================================================

import mlflow
from mlflow.models import infer_signature
from mlflow.models.resources import DatabricksServingEndpoint
import pandas as pd
import json


class CareMapModelFixed(mlflow.pyfunc.PythonModel):
    """Fixed MLflow model that works in Model Serving environment"""
    
    def __init__(self):
        self.client = None
        self.facilities_data = None
    
    def load_context(self, context):
        """Initialize - no PySpark, load cached facilities data"""
        import mlflow.deployments
        
        print("🔄 Initializing CareMap model (no PySpark)...")
        
        # Initialize LLM client only
        self.client = mlflow.deployments.get_deploy_client("databricks")
        
        # Load pre-cached facilities data from artifacts
        import pickle
        facilities_path = context.artifacts.get("facilities_data")
        if facilities_path:
            with open(facilities_path, 'rb') as f:
                self.facilities_data = pickle.load(f)
            print(f"✅ Loaded {len(self.facilities_data)} facilities from cache")
        else:
            print("⚠️ No facilities data cached - will return empty results")
            self.facilities_data = []
        
        print("✅ CareMap model ready")
    
    def predict(self, context, model_input):
        """Handle API requests"""
        questions = model_input["question"].tolist()
        
        results = []
        for question in questions:
            try:
                response = self._caremap_pipeline(question)
                results.append(response)
            except Exception as e:
                results.append({
                    "question": question,
                    "error": str(e),
                    "careguide_message": f"Error: {str(e)}"
                })
        
        return pd.DataFrame(results)
    
    def _caremap_pipeline(self, user_question):
        """Simplified pipeline using cached data"""
        
        # Step 1: Query Agent
        intent = self._query_agent(user_question)
        
        # Step 2: Search Agent (using cached data)
        candidates = self._search_agent_cached(intent)
        
        # Step 3: Validator Agent
        validated = self._validator_agent(user_question, intent, candidates)
        
        # Build response
        return {
            "question": user_question,
            "intent": json.dumps(intent),
            "candidates_found": len(candidates),
            "careguide_message": validated.get("answer", ""),
            "emergency_note": validated.get("emergency_note"),
            "facility_cards": json.dumps(validated.get("validated_results", [])),
            "chain_of_thought": json.dumps(validated.get("reasoning_steps", [])),
            "facilities": json.dumps([{
                "name": c["facility_name"],
                "city": c["city"],
                "state": c["state"],
                "lat": c["latitude"],
                "lng": c["longitude"],
                "trust_score": c["trust_score"]
            } for c in candidates[:10]])
        }
    
    def _query_agent(self, user_question):
        """Same as before - parses user question"""
        prompt = f"""You are a healthcare facility search assistant for India.
Parse the user's question into a structured JSON query.

Extract these fields:
- "search_text": the core medical need
- "state": Indian state filter (null if not mentioned)
- "city": city filter (null if not mentioned)
- "facility_type": hospital/clinic/dentist (null if not mentioned)
- "required_capabilities": list from [emergency, surgery, icu, dialysis, neonatal]
- "specialties_needed": list of medical specialties
- "min_trust_score": 60 if user wants trusted facilities, else 0
- "urgency": high/medium/low

Respond with ONLY valid JSON.

User Question: {user_question}"""
        
        response = self.client.predict(
            endpoint="databricks-meta-llama-3-3-70b-instruct",
            inputs={"messages": [{"role": "user", "content": prompt}], "max_tokens": 500, "temperature": 0.1}
        )
        
        try:
            result_text = response['choices'][0]['message']['content']
            result_text = result_text.replace("```json", "").replace("```", "").strip()
            return json.loads(result_text)
        except:
            return {
                "search_text": user_question,
                "state": None, "city": None, "facility_type": None,
                "required_capabilities": [], "specialties_needed": [],
                "min_trust_score": 0, "urgency": "medium"
            }
    
    def _search_agent_cached(self, query_intent):
        """Search using cached facilities data - no PySpark needed"""
        
        # Filter cached data
        results = self.facilities_data
        
        # Apply filters
        state = query_intent.get("state")
        if state:
            results = [f for f in results if f.get("state") == state]
        
        city = query_intent.get("city")
        if city:
            results = [f for f in results if f.get("city") == city]
        
        min_trust = query_intent.get("min_trust_score", 0)
        if min_trust > 0:
            results = [f for f in results if f.get("trust_score", 0) >= min_trust]
        
        # Capability filters
        required_caps = query_intent.get("required_capabilities", [])
        for cap in required_caps:
            cap_key = f"has_{cap}"
            results = [f for f in results if f.get(cap_key) == True]
        
        # Sort by trust score and limit
        results = sorted(results, key=lambda x: x.get("trust_score", 0), reverse=True)[:10]
        
        return results
    
    def _validator_agent(self, user_question, query_intent, candidates):
        """Same as before - validates and recommends"""
        if not candidates:
            return {
                "answer": "No facilities found. Call 108 if emergency. — CareGuide",
                "validated_results": [],
                "reasoning_steps": []
            }
        
        # Build facility summaries
        facility_summaries = ""
        for i, c in enumerate(candidates[:5]):
            facility_summaries += f"""
FACILITY {i+1}: {c['facility_name']}
- Location: {c['city']}, {c['state']}
- Trust Score: {c['trust_score']}/100
- Emergency: {c.get('has_emergency')} | Surgery: {c.get('has_surgery')}
"""
        
        prompt = f"""You are CareGuide for CareMap.

User asked: "{user_question}"
Facilities: {facility_summaries}

Provide warm recommendation. JSON format:
{{
    "answer": "Recommendation. — CareGuide, by CareMap",
    "emergency_note": "Call 108 if urgent" or null,
    "validated_results": [
        {{"rank": 1, "facility_name": "name", "city": "city", "state": "state",
          "latitude": 0.0, "longitude": 0.0, "verdict": "VERIFIED & RECOMMENDED",
          "what_they_offer": "Services", "concerns": "Concerns or 'Verified'"}}
    ],
    "reasoning_steps": ["Step 1: ..."]
}}

ONLY JSON."""
        
        response = self.client.predict(
            endpoint="databricks-meta-llama-3-3-70b-instruct",
            inputs={"messages": [{"role": "user", "content": prompt}], "max_tokens": 2000, "temperature": 0.3}
        )
        
        try:
            result_text = response['choices'][0]['message']['content']
            result_text = result_text.replace("```json", "").replace("```", "").strip()
            return json.loads(result_text)
        except:
            return {
                "answer": "Found facilities but processing failed. — CareGuide",
                "validated_results": [],
                "reasoning_steps": []
            }


# ============================================================================
# PREPARE AND LOG FIXED MODEL
# ============================================================================

print("📦 Preparing fixed model with cached data...\n")

# Export current facilities data to cache
import pickle
import tempfile
import os

# Get facilities from current session
facilities_df = spark.sql("""
    SELECT 
        name_cleaned as facility_name,
        city, state, facilityType as facility_type,
        trust_score, trust_label, completeness_score,
        has_emergency, has_surgery, has_icu, has_dialysis, has_neonatal,
        description_cleaned as description,
        flag_surgery_no_surgeon, flag_icu_no_equipment,
        flag_emergency_no_basics, flag_specialty_no_specialist,
        latitude_valid as latitude, longitude_valid as longitude
    FROM healthcare_facilities_cleaned
    LIMIT 1000
""").toPandas()

facilities_list = facilities_df.to_dict('records')
print(f"✅ Exported {len(facilities_list)} facilities to cache")

# Create temporary file for facilities data
temp_dir = tempfile.mkdtemp()
facilities_cache_path = os.path.join(temp_dir, "facilities_data.pkl")
with open(facilities_cache_path, 'wb') as f:
    pickle.dump(facilities_list, f)

print(f"✅ Cached facilities to: {facilities_cache_path}")

# Create sample signature
sample_input = pd.DataFrame({"question": ["Find hospitals in Bihar"]})
sample_output = pd.DataFrame([{
    "question": "Find hospitals in Bihar",
    "intent": "{}",
    "candidates_found": 0,
    "careguide_message": "Sample",
    "emergency_note": None,
    "facility_cards": "[]",
    "chain_of_thought": "[]",
    "facilities": "[]"
}])

signature = infer_signature(sample_input, sample_output)

# Log model with artifacts AND resources
print("\n📦 Logging model to MLflow with resource dependencies...")

with mlflow.start_run(run_name="caremap_api_fixed_v3") as run:
    mlflow.pyfunc.log_model(
        artifact_path="caremap_model",
        python_model=CareMapModelFixed(),
        artifacts={"facilities_data": facilities_cache_path},
        signature=signature,
        registered_model_name="workspace.default.caremap_api_fixed",
        pip_requirements=[
            "mlflow>=2.9.0",
            "pandas",
            "requests"
        ],
        resources=[
            DatabricksServingEndpoint(endpoint_name="databricks-meta-llama-3-3-70b-instruct")
        ],
        await_registration_for=180
    )
    
    print(f"✅ Model logged successfully with resource dependencies")
    print(f"✅ Registered as: workspace.default.caremap_api_fixed version 3")

print("\n" + "="*60)
print("🚀 Ready to deploy! Update Cell 15 entity_version to 3 and run.")
print("="*60)

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8c1ed9f8-e9ce-4dc5-a509-abf06a45e5c6/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


📦 Preparing fixed model with cached data...

✅ Exported 1000 facilities to cache
✅ Cached facilities to: /tmp/tmpx9tuq6le/facilities_data.pkl

📦 Logging model to MLflow with resource dependencies...


/local_disk0/.ephemeral_nfs/envs/pythonEnv-8c1ed9f8-e9ce-4dc5-a509-abf06a45e5c6/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 11:35:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-90e1d16e-465c.cloud.databricks.com/m

Registered model 'workspace.default.caremap_api_fixed' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '3' of model 'workspace.default.caremap_api_fixed': https://dbc-90e1d16e-465c.cloud.databricks.com/explore/data/models/workspace/default/caremap_api_fixed/version/3?o=1545005666056481


✅ Model logged successfully with resource dependencies
✅ Registered as: workspace.default.caremap_api_fixed version 3

🚀 Ready to deploy! Update Cell 15 entity_version to 3 and run.


In [0]:
# ============================================================================
# DEPLOY FIXED MODEL
# ============================================================================

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ServedEntityInput,
    EndpointCoreConfigInput
)

w = WorkspaceClient()

endpoint_name = "caremap-api"
model_name = "workspace.default.caremap_api_fixed"

print(f"🚀 Updating endpoint to serve model version 3 (with proper resource dependencies)")
print("   This may take 5-10 minutes...\n")

try:
    # Update existing endpoint to use version 3
    endpoint = w.serving_endpoints.update_config_and_wait(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                entity_name=model_name,
                entity_version="3",
                scale_to_zero_enabled=True,
                workload_size="Small"
            )
        ]
    )
    
    print(f"✅ Endpoint updated successfully!\n")
    print(f"🎉 Now serving version 3 with DatabricksServingEndpoint resources\n")
    print(f"🌐 API URL:")
    
    # Get workspace URL
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
    api_url = f"https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations"
    print(f"   {api_url}")
    
    print(f"\n📝 Test it with Cell 16!")
    
except Exception as e:
    print(f"❌ Error: {e}")

🚀 Updating endpoint to serve model version 3 (with proper resource dependencies)
   This may take 5-10 minutes...

✅ Endpoint updated successfully!

🎉 Now serving version 3 with DatabricksServingEndpoint resources

🌐 API URL:
   https://dbc-90e1d16e-465c.cloud.databricks.com/serving-endpoints/caremap-api/invocations

📝 Test it with Cell 16!


In [0]:
# ============================================================================
# TEST THE DEPLOYED API
# ============================================================================

import requests
import json

# Get workspace URL and token
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

endpoint_name = "caremap-api"
api_url = f"https://{workspace_url}/serving-endpoints/{endpoint_name}/invocations"

print(f"🧪 Testing API: {api_url}\n")

# Test queries
test_questions = [
    "Find emergency surgery facility in Bihar",
    "Show me trusted eye hospitals in Hyderabad",
    "My mother needs neonatal care urgently"
]

for question in test_questions:
    print(f"👉 Question: {question}")
    print("-" * 60)
    
    try:
        response = requests.post(
            api_url,
            headers={
                "Authorization": f"Bearer {token}",
                "Content-Type": "application/json"
            },
            json={
                "inputs": {
                    "question": [question]
                }
            },
            timeout=60
        )
        
        if response.status_code == 200:
            result = response.json()
            predictions = result.get("predictions", [])
            
            if predictions:
                pred = predictions[0]
                print(f"✅ Response received:")
                print(f"   Candidates found: {pred.get('candidates_found', 0)}")
                print(f"   CareGuide says: {pred.get('careguide_message', '')[:150]}...")
                
                # Parse facility cards
                try:
                    cards = json.loads(pred.get('facility_cards', '[]'))
                    print(f"\n   🏞️ Top {len(cards)} Facilities:")
                    for card in cards[:3]:
                        print(f"      • {card.get('facility_name')} - {card.get('verdict')}")
                except:
                    pass
            else:
                print(f"⚠️ No predictions in response")
        else:
            print(f"❌ Error {response.status_code}: {response.text[:200]}")
    
    except Exception as e:
        print(f"❌ Request failed: {str(e)[:200]}")
    
    print("\n" + "="*60 + "\n")

print("✅ API testing complete!")

🧪 Testing API: https://dbc-90e1d16e-465c.cloud.databricks.com/serving-endpoints/caremap-api/invocations

👉 Question: Find emergency surgery facility in Bihar
------------------------------------------------------------
✅ Response received:
   Candidates found: 4
   CareGuide says: Based on your request, I recommend A.R DENTAL EXPERT - Kishanganj Branch for emergency surgery in Bihar. — CareGuide, by CareMap...

   🏞️ Top 1 Facilities:
      • A.R DENTAL EXPERT - Kishanganj Branch - VERIFIED & RECOMMENDED


👉 Question: Show me trusted eye hospitals in Hyderabad
------------------------------------------------------------
✅ Response received:
   Candidates found: 0
   CareGuide says: Error: 'NoneType' object is not iterable...

   🏞️ Top 0 Facilities:


👉 Question: My mother needs neonatal care urgently
------------------------------------------------------------
✅ Response received:
   Candidates found: 7
   CareGuide says: Unfortunately, none of the facilities listed offer emergency se

In [0]:
import requests
import json

workspace_url = "https://dbc-90e1d16e-465c.cloud.databricks.com"
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

response = requests.post(
    f"{workspace_url}/serving-endpoints/caremap-api/invocations",
    headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
    json={"inputs": {"question": ["Find emergency surgery facility in Bihar"]}}
)

print(json.dumps(response.json(), indent=2))

{
  "predictions": [
    {
      "question": "Find emergency surgery facility in Bihar",
      "intent": "{\"search_text\": \"surgery\", \"state\": \"Bihar\", \"city\": null, \"facility_type\": \"hospital\", \"required_capabilities\": [\"emergency\", \"surgery\"], \"specialties_needed\": [], \"min_trust_score\": 0, \"urgency\": \"high\"}",
      "candidates_found": 4,
      "careguide_message": "Based on your request, I recommend A.R DENTAL EXPERT - Kishanganj Branch for emergency surgery in Bihar. \u2014 CareGuide, by CareMap",
      "emergency_note": "Call 108 if urgent",
      "facility_cards": "[{\"rank\": 1, \"facility_name\": \"A.R DENTAL EXPERT - Kishanganj Branch\", \"city\": \"Kishanganj\", \"state\": \"Bihar\", \"latitude\": 0.0, \"longitude\": 0.0, \"verdict\": \"VERIFIED & RECOMMENDED\", \"what_they_offer\": \"Emergency and Surgery services\", \"concerns\": \"Verified\"}]",
      "chain_of_thought": "[\"Step 1: Filtered facilities in Bihar that offer emergency and surgery s

In [0]:
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
print("Bearer [REDACTED]")

Bearer [REDACTED]
